# RADAR — `synthetic_data_generator` recipes

This notebook shows how to **actually train** the fixed RADAR reference model on your own
*synthetic data generators* and score them on GIFT-Eval — the exact path the validator runs
for the `synthetic_data_generator` task, driven by hand so you can iterate on generators.

**What a generator is.** You write one function:

```python
def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    # return an iterator that yields batches forever
    while True:
        yield {"input": x, "target": y}   # x:(B, context_len, V)  y:(B, prediction_len, V)
```

The validator trains the *same* miniature (~10M-param) Toto-2.0-style causal patch decoder on
whatever data your generator emits, then scores the trained checkpoint on the full GIFT-Eval
leaderboard. The architecture never changes — **you compete purely on the data**.

Score: `metric = sqrt(crps * mase)` (lower is better). The in-training val curve is diagnostics
only.

### What you get here
1. Environment + (optional) GIFT-Eval / val-shard credentials.
2. A zero-creds **smoke test** for iterating on generator shapes.
3. Four worked generator recipes (AR(1) baseline → rich mixture → regime sampler → GP draws).
4. A driver that runs the real train+eval and plots the loss curves.
5. A **continuation** recipe (warm-start from a previous checkpoint).

## 0. Runtime

Use a **GPU runtime** (Runtime → Change runtime type → GPU). Training works on CPU but is slow.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. Clone the repo and install

The `[ts_forecasting]` extra pulls torch + safetensors + pyarrow + boto3 — everything the real
train/eval path needs. (Colab already ships torch; pip will keep the existing build.)

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/tensorlink-ai/radar.git'
if not os.path.isdir('radar'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)

os.chdir('/content/radar' if os.path.isdir('/content/radar') else 'radar')
REPO_ROOT = os.getcwd()
print('repo root:', REPO_ROOT)

# Editable install with the heavy extra. Safe to re-run.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ts_forecasting]'], check=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('installed')

## 2. (Optional) Credentials for real data

Two caches feed the real pipeline, both pulled from the S3-compatible Hippius/R2 store:

| Cache | Env var | What it's for | Needed for |
|---|---|---|---|
| GIFT-Eval benchmark | `RADAR_GIFT_EVAL_CACHE` | the held-out leaderboard | the **score** (`crps`/`mase`) |
| Pretrain val shard | `RADAR_PRETRAIN_VAL_CACHE` | fixed in-training val split | the **val loss curve** (diagnostics) |

Your **generator is the only training source** — training shards are *never* downloaded for this
task. So:

- **No creds?** You can still run every generator through real training and watch the train-loss
  curve (Section 3 + the smoke test). GIFT scoring will report a failure because the benchmark
  cache is absent — that's expected offline.
- **With creds?** You get the real `sqrt(crps*mase)` score and a val curve too.

Fill in the cell below if you have keys (Hippius **or** legacy R2). Leave blank to run offline.

In [ ]:
# --- Fill these in for real GIFT-Eval scoring (leave blank to run offline) ---
os.environ.setdefault('HIPPIUS_ACCESS_KEY_ID',     '')   # or R2_ACCESS_KEY_ID
os.environ.setdefault('HIPPIUS_SECRET_ACCESS_KEY', '')   # or R2_SECRET_ACCESS_KEY
# Endpoint defaults to https://s3.hippius.com. For Cloudflare R2 instead set:
#   os.environ['R2_ACCOUNT_ID'] = '...'   (derives the per-account endpoint, region=auto)

# Local cache dirs (defaults match the validator).
os.environ.setdefault('RADAR_GIFT_EVAL_CACHE',   '/tmp/radar_gift_eval')
os.environ.setdefault('RADAR_PRETRAIN_VAL_CACHE', '/tmp/radar_pretrain_val')

HAVE_CREDS = bool(os.environ.get('HIPPIUS_ACCESS_KEY_ID') or os.environ.get('R2_ACCESS_KEY_ID'))
print('credentials present:', HAVE_CREDS)
print('gift cache :', os.environ['RADAR_GIFT_EVAL_CACHE'])
print('val cache  :', os.environ['RADAR_PRETRAIN_VAL_CACHE'])

### 2a. Fetch the caches (only if you set credentials)

`fetch_gift_eval` downloads Arrow benchmark files; `fetch_pretrain --val` grabs the single fixed
held-out val shard. The full GIFT-Eval benchmark is large — for a first run pull a small subset of
datasets so the eval is quick. (The validator always scores all 97 tasks; a subset just makes the
notebook fast and is fine for experimenting.)

In [ ]:
if HAVE_CREDS:
    # A few quick datasets to keep the eval fast while you iterate.
    # Drop the --datasets line (use --all) to mirror the full leaderboard.
    subprocess.run([sys.executable, '-m', 'local.fetch_gift_eval',
                    '--datasets', 'm4_hourly', 'm4_daily', 'm4_weekly'], check=False)
    subprocess.run([sys.executable, '-m', 'local.fetch_pretrain', '--val'], check=False)
    import glob
    print('gift files:', len(glob.glob(os.environ['RADAR_GIFT_EVAL_CACHE'] + '/**/*.arrow', recursive=True)))
    print('val files :', len(glob.glob(os.environ['RADAR_PRETRAIN_VAL_CACHE'] + '/*.parquet')))
else:
    print('No creds set — skipping downloads. Training still runs; GIFT scoring will be skipped.')

## 3. The generator contract + a zero-creds smoke test

Before burning a training budget, sanity-check that a generator yields the right shapes. The
validator runs every batch through `_ValidatingBatchIter`, which enforces:

- batch is a `dict` with `input`/`target` (or a `(input, target)` tuple),
- `input` is `(B, context_len, num_variates)`, `target` is `(B, prediction_len, num_variates)`,
- same batch dim on both. Non-finite values (nan/inf) are sanitized to 0, not fatal.

The GIFT-Eval constants the validator passes in:
`context_len=512`, `prediction_len=96`, `num_variates=1`, `quantiles=[0.1..0.9]`.

In [ ]:
import numpy as np, torch

CONTEXT_LEN, PREDICTION_LEN, NUM_VARIATES = 512, 96, 1
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def smoke_test(generator_code, n_batches=2):
    """Exec a generator string and check its first few batches."""
    ns = {'__name__': 'smoke'}
    exec(generator_code, ns)
    bp = ns.get('build_pipeline')
    assert callable(bp), 'generator must define build_pipeline(...)'
    it = bp(CONTEXT_LEN, PREDICTION_LEN, NUM_VARIATES, QUANTILES)
    batches = []
    for i, batch in zip(range(n_batches), it):
        if isinstance(batch, (tuple, list)):
            x, y = batch
        else:
            x, y = batch.get('input', batch.get('context')), batch['target']
        x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
        y = torch.as_tensor(np.asarray(y), dtype=torch.float32)
        assert x.dim() == 3 and x.shape[1:] == (CONTEXT_LEN, NUM_VARIATES), f'bad input shape {tuple(x.shape)}'
        assert y.dim() == 3 and y.shape[1:] == (PREDICTION_LEN, NUM_VARIATES), f'bad target shape {tuple(y.shape)}'
        assert x.shape[0] == y.shape[0], 'batch dim mismatch'
        finite = float(torch.isfinite(x).float().mean()), float(torch.isfinite(y).float().mean())
        print(f'batch {i}: input {tuple(x.shape)} target {tuple(y.shape)} finite(in/out)={finite}')
        batches.append((x, y))
    print('OK')
    return batches

## Recipe 1 — AR(1) random walks (the baseline)

This is the validator's own reference pipeline: correlated random walks (`rho=0.95`). Enough
structure that the val curve isn't flat, but no task-specific signal — a good sanity floor and a
template for the contract.

In [ ]:
RECIPE_AR1 = '''
import torch

def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    g = torch.Generator().manual_seed(0xBA5E11AE)
    batch_size = 32
    total = int(context_len) + int(prediction_len)
    rho = 0.95
    while True:
        noise = torch.randn(batch_size, total, int(num_variates), generator=g)
        series = torch.empty_like(noise)
        series[:, 0, :] = noise[:, 0, :]
        for t in range(1, total):
            series[:, t, :] = rho * series[:, t - 1, :] + noise[:, t, :]
        yield {
            "input":  series[:, :int(context_len), :],
            "target": series[:, int(context_len):, :],
        }
'''

smoke_test(RECIPE_AR1);

## Recipe 2 — Trend + multi-seasonal + noise mixture

A richer synthetic prior closer to real time series: each series is a random linear/quadratic
trend, **two** seasonal components (random periods/phases/amplitudes), plus heteroscedastic noise.
Normalised per series so the robust scaler sees sane magnitudes. This kind of structured prior
tends to transfer better to GIFT-Eval than pure random walks.

In [ ]:
RECIPE_SEASONAL = '''
import torch, math

def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    g = torch.Generator().manual_seed(1234)
    B, V = 64, int(num_variates)
    total = int(context_len) + int(prediction_len)
    t = torch.arange(total, dtype=torch.float32).view(1, total, 1)
    while True:
        # trend: random slope + mild curvature
        slope = torch.randn(B, 1, V, generator=g) * 0.01
        curve = torch.randn(B, 1, V, generator=g) * 1e-5
        trend = slope * t + curve * t * t
        # two seasonal components with random period/phase/amplitude
        series = trend.clone()
        for _ in range(2):
            period = torch.randint(5, 200, (B, 1, V), generator=g).float()
            phase  = torch.rand(B, 1, V, generator=g) * 2 * math.pi
            amp    = torch.rand(B, 1, V, generator=g) * 2.0 + 0.2
            series = series + amp * torch.sin(2 * math.pi * t / period + phase)
        # heteroscedastic noise
        noise_scale = torch.rand(B, 1, V, generator=g) * 0.5 + 0.05
        series = series + noise_scale * torch.randn(B, total, V, generator=g)
        # per-series normalise (mean/std over context window)
        ctx = series[:, :int(context_len), :]
        mu = ctx.mean(dim=1, keepdim=True)
        sd = ctx.std(dim=1, keepdim=True).clamp_min(1e-6)
        series = (series - mu) / sd
        yield {
            "input":  series[:, :int(context_len), :],
            "target": series[:, int(context_len):, :],
        }
'''

smoke_test(RECIPE_SEASONAL);

## Recipe 3 — Regime sampler (mixture of priors)

GIFT-Eval spans many domains, so a *diverse* training distribution usually beats a single prior.
Each batch randomly picks one of several regimes — AR(1), seasonal, piecewise-linear with level
shifts, and spiky/intermittent — so the model sees broad structure. This is a strong, simple
template to extend with your own regimes.

In [ ]:
RECIPE_REGIME = '''
import torch, math

def _ar1(B, total, V, g):
    rho = 0.7 + 0.29 * torch.rand(1, generator=g).item()
    noise = torch.randn(B, total, V, generator=g)
    s = torch.empty_like(noise); s[:, 0, :] = noise[:, 0, :]
    for t in range(1, total):
        s[:, t, :] = rho * s[:, t - 1, :] + noise[:, t, :]
    return s

def _seasonal(B, total, V, g):
    t = torch.arange(total, dtype=torch.float32).view(1, total, 1)
    period = torch.randint(5, 150, (B, 1, V), generator=g).float()
    phase  = torch.rand(B, 1, V, generator=g) * 2 * math.pi
    amp    = torch.rand(B, 1, V, generator=g) * 2 + 0.5
    return amp * torch.sin(2 * math.pi * t / period + phase) + 0.2 * torch.randn(B, total, V, generator=g)

def _piecewise(B, total, V, g):
    s = torch.zeros(B, total, V)
    n_seg = 4
    bounds = torch.linspace(0, total, n_seg + 1).long()
    level = torch.zeros(B, 1, V)
    for i in range(n_seg):
        a, b = bounds[i].item(), bounds[i + 1].item()
        slope = torch.randn(B, 1, V, generator=g) * 0.05
        step  = torch.randn(B, 1, V, generator=g) * 2.0  # level shift
        local_t = torch.arange(b - a, dtype=torch.float32).view(1, -1, 1)
        level = level + step
        s[:, a:b, :] = level + slope * local_t
    return s + 0.1 * torch.randn(B, total, V, generator=g)

def _spiky(B, total, V, g):
    base = 0.2 * torch.randn(B, total, V, generator=g)
    mask = (torch.rand(B, total, V, generator=g) < 0.05).float()
    spikes = mask * torch.randn(B, total, V, generator=g) * 5.0
    return base + spikes

def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    g = torch.Generator().manual_seed(7)
    B, V = 64, int(num_variates)
    total = int(context_len) + int(prediction_len)
    regimes = [_ar1, _seasonal, _piecewise, _spiky]
    while True:
        fn = regimes[torch.randint(len(regimes), (1,), generator=g).item()]
        series = fn(B, total, V, g)
        ctx = series[:, :int(context_len), :]
        mu = ctx.mean(dim=1, keepdim=True)
        sd = ctx.std(dim=1, keepdim=True).clamp_min(1e-6)
        series = (series - mu) / sd
        yield {
            "input":  series[:, :int(context_len), :],
            "target": series[:, int(context_len):, :],
        }
'''

smoke_test(RECIPE_REGIME);

## Recipe 4 — Gaussian-process draws (smooth structure)

Samples from a GP with an RBF kernel (random length-scale) give smooth, correlated series with no
explicit periodicity — a complementary prior to the seasonal/AR families. We sample per batch from
a Cholesky factor of the kernel; keep `context+horizon` modest so the covariance stays cheap.

In [ ]:
RECIPE_GP = '''
import torch

def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    g = torch.Generator().manual_seed(99)
    B, V = 32, int(num_variates)
    total = int(context_len) + int(prediction_len)
    t = torch.arange(total, dtype=torch.float32)
    d2 = (t.view(-1, 1) - t.view(1, -1)) ** 2  # (total, total)
    while True:
        lengthscale = torch.empty(1, generator=g).uniform_(10.0, 80.0).item()
        K = torch.exp(-d2 / (2 * lengthscale ** 2)) + 1e-4 * torch.eye(total)
        L = torch.linalg.cholesky(K)
        z = torch.randn(total, B * V, generator=g)
        series = (L @ z).T.reshape(B, V, total).permute(0, 2, 1).contiguous()
        yield {
            "input":  series[:, :int(context_len), :],
            "target": series[:, int(context_len):, :],
        }
'''

smoke_test(RECIPE_GP);

## 4. Train + evaluate a generator

`run_synth_generator_training` is exactly what the validator calls. It execs your generator, trains
the fixed reference arch under a time budget, saves the best-val checkpoint, then runs GIFT-Eval.

- `min_flops=max_flops=0` disables the size gate (the arch is fixed, so we don't gate it).
- `task.time_budget_seconds` is the train budget — keep it short here to iterate.

**Offline note:** with no benchmark cache the result is `success=False` with a GIFT-cache error —
but `loss_curve`/`val_curve` are still populated, so you can compare generators by their training
dynamics. With creds you also get `objectives['crps']`, `['mase']`, and the final `metric`.

In [ ]:
from types import SimpleNamespace
from local.synth_generator import run_synth_generator_training

def train_generator(code, *, seed=0, budget_seconds=120, parent_checkpoint=None,
                    compute_offset=0.0, step_offset=0):
    task = SimpleNamespace(time_budget_seconds=int(budget_seconds))
    result = run_synth_generator_training(
        code,
        seed=seed,
        task=task,
        min_flops=0,
        max_flops=0,
        parent_checkpoint_path=parent_checkpoint,
        compute_offset=compute_offset,
        step_offset=step_offset,
    )
    print('success :', result['success'])
    print('metric  :', result['metric'], '(sqrt(crps*mase), lower=better)')
    obj = result['objectives']
    for k in ('crps', 'mase', 'best_val_loss', 'this_compute', 'num_params'):
        if k in obj:
            print(f'  {k:14s}: {obj[k]}')
    if not result['success']:
        print('note    :', result.get('error', '')[:200])
    return result

In [ ]:
# Train the AR(1) baseline for a short budget. Bump budget_seconds for a real run.
res_ar1 = train_generator(RECIPE_AR1, seed=0, budget_seconds=120)

### Plot the training / val curves

In [ ]:
import matplotlib.pyplot as plt

def plot_curves(result, label=''):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    lc = result.get('loss_curve') or []
    ax[0].plot(lc); ax[0].set_title(f'train loss {label}'); ax[0].set_xlabel('log step'); ax[0].grid(alpha=.3)
    vc = result.get('val_curve') or []
    if vc:
        ax[1].plot([p['step'] for p in vc], [p['loss'] for p in vc], 'o-')
        ax[1].set_title(f'val loss {label}'); ax[1].set_xlabel('step'); ax[1].grid(alpha=.3)
    else:
        ax[1].text(0.5, 0.5, 'no val curve\n(set RADAR_PRETRAIN_VAL_CACHE)', ha='center', va='center')
    plt.tight_layout(); plt.show()

plot_curves(res_ar1, 'AR(1)')

### Compare a few recipes

Run several generators with the same seed + budget and line them up. With GIFT creds, sort by
`metric`; offline, compare final train/val loss.

In [ ]:
recipes = {
    'ar1':      RECIPE_AR1,
    'seasonal': RECIPE_SEASONAL,
    'regime':   RECIPE_REGIME,
    'gp':       RECIPE_GP,
}

results = {}
for name, code in recipes.items():
    print('=== training', name, '===')
    results[name] = train_generator(code, seed=0, budget_seconds=120)
    print()

print('name      metric        final_train   best_val')
for name, r in results.items():
    lc = r.get('loss_curve') or [None]
    print(f"{name:9s} {str(r['metric']):13s} {str(lc[-1]):13s} {r['objectives'].get('best_val_loss')}")

## 5. Continuation — warm-start from a previous checkpoint

A continuation round keeps training the **same weights** on (hopefully better) data instead of
starting fresh. The validator scores continuations on a second frontier: cumulative compute vs the
GIFT-Eval improvement over the parent. Here we just show the mechanic — grab a parent checkpoint
and warm-start a second run from it.

The saved checkpoint lives at `<workdir>/checkpoints/model.safetensors` from the parent run.

In [ ]:
from pathlib import Path

parent = res_ar1                       # any prior result with success or at least a saved ckpt
parent_ckpt = Path(parent['workdir']) / 'checkpoints' / 'model.safetensors'
print('parent checkpoint exists:', parent_ckpt.exists(), '->', parent_ckpt)

if parent_ckpt.exists():
    # Offset the curve coordinates into lineage-absolute space so the stitched
    # trajectory is monotonic (optional but matches the validator).
    compute_offset = float(parent['objectives'].get('cumulative_compute', 0.0))
    res_cont = train_generator(
        RECIPE_SEASONAL,                # continue with a (maybe better) generator
        seed=0,
        budget_seconds=120,
        parent_checkpoint=str(parent_ckpt),
        compute_offset=compute_offset,
    )
    plot_curves(res_cont, 'continuation')
else:
    print('No parent checkpoint — run a parent training cell first.')

## 6. Write your own

Start from the smoke test, then train. A few things that tend to help on GIFT-Eval:

- **Diversity beats any single prior** — mix trends, multiple seasonalities, level shifts, spikes,
  intermittency, and varied noise (see Recipe 3).
- **Per-series normalisation** keeps magnitudes in a range the arcsinh robust scaler likes.
- **Match the horizon** — emit `prediction_len` future steps; the harness aligns/pads but clean
  targets train better.
- **Non-finite values are tolerated** (sanitised to 0) but a flood of them wastes the budget.
- **Reproducibility** — seed your `torch.Generator` so a given generator is comparable run to run.

When you're happy with a generator, the same code string is what a miner submits as
`build_pipeline` for the `synthetic_data_generator` task.

In [ ]:
MY_RECIPE = '''
import torch

def build_pipeline(context_len, prediction_len, num_variates, quantiles):
    g = torch.Generator().manual_seed(0)
    total = int(context_len) + int(prediction_len)
    while True:
        # TODO: design your synthetic data here.
        series = torch.randn(64, total, int(num_variates), generator=g)
        yield {
            "input":  series[:, :int(context_len), :],
            "target": series[:, int(context_len):, :],
        }
'''

smoke_test(MY_RECIPE)
# res_mine = train_generator(MY_RECIPE, seed=0, budget_seconds=120)
# plot_curves(res_mine, 'mine')